# Exploring and preparing retail data (SP3 rehearsal)

This is a rehearsal of the official 2027 sample problem SP3 ("Exploring and preparing retail data"), with synthetic data and the same rules.

**How this maps to the real exam.** In the exam you work in a Colab notebook: the starter code downloads `transactions.csv` and `products.csv` from the organizer's server, you type the Part A and Part C answers into answer boxes (rounded with `round(value, 4)`), and you upload the two CSV files your notebook writes. Here the same work is split into five functions in one file, so the site can grade it: `explore` is Part A, `clean` is Part B, `analyze` is Part C, `customer_summary` is Part D, and `save_outputs` writes the files. The grader generates the data, loads it exactly as the exam does, calls your functions, and compares the results with its own independent computation. The exam has 20 seconds of CPU time for everything after the starter code; so does this file.

## Data

Both tables are loaded with `pd.read_csv(path, dtype=str, keep_default_na=False)`: every value is a string, and an empty cell is `""`.

- `transactions`: columns `record_id, invoice_id, product_code, description, quantity, unit_price, customer_id, country`. A record can appear more than once. Text can carry extra spaces and mixed case. Quantities and prices can be empty, non-numeric, nonpositive, or (quantities) non-integer.
- `products`: columns `product_code, category`. Each product code appears at most once; some transaction products are missing from it.

## Part A: `explore(transactions, products) -> dict`

Computed on the raw `transactions` table. "Numeric" means `pd.to_numeric(value, errors="coerce")` gives a number. Return a dict with exactly these keys:

| key | value |
|---|---|
| `"rows"`, `"columns"` | number of rows and columns of `transactions` |
| `"distinct_customers"` | distinct customer IDs after `strip()`, excluding the empty ID |
| `"duplicate_copies"` | extra copies: a `record_id` that appears $k$ times counts $k - 1$ |
| `"nonnumeric_quantity"` | rows whose `quantity` is not `""` but is not numeric |
| `"nonpositive_price"` | rows whose `unit_price` is numeric and $\le 0$ |
| `"median_positive_price"` | median of the numeric unit prices that are $> 0$ |
| `"pct_missing_customer"` | percent (0 to 100) of rows whose customer ID is empty after `strip()` |
| `"cancellation_rows"` | rows whose `invoice_id` starts with `C`, ignoring case and surrounding whitespace |

## Part B: `clean(transactions, products) -> DataFrame`

Apply these steps **in this order**:

1. `strip()` the columns `invoice_id`, `product_code`, `customer_id`, `country` (not `description`). Then lowercase `country`, replace the exact value `"uk"` by `"united kingdom"` and `""` by `"unknown"`, then apply `str.title()`.
2. Convert `quantity` and `unit_price` with `pd.to_numeric(..., errors="coerce")`.
3. Keep only the first occurrence of each `record_id` (file order).
4. Drop rows where any of these holds: `invoice_id` is empty; `product_code` is empty; `invoice_id` starts with `C` (ignoring case); `quantity` is missing, $\le 0$, or not a whole number; `unit_price` is present and $\le 0$. Keep rows whose `unit_price` is missing.
5. For each `product_code`, compute the median of the remaining non-missing unit prices (all medians are computed before any filling, unrounded). Fill each missing unit price with its product's median. Drop rows whose price is still missing. Add `price_imputed`: 1 if the price was filled in this step, else 0.
6. Left-join `products` on `product_code`. A product with no category gets `"Unclassified"`.
7. Add `line_amount = quantity * unit_price` and `quantity_band`: `"Small"` for quantity 1 to 5, `"Medium"` for 6 to 20, `"Large"` above 20.
8. Return exactly the columns `record_id, invoice_id, product_code, description, quantity, unit_price, customer_id, country, category, price_imputed, line_amount, quantity_band`, sorted by `invoice_id`, then `product_code`, then `record_id`, all compared **as strings**.

Dtypes: `quantity` and `price_imputed` are int64; `unit_price` and `line_amount` are float64; all other columns are strings. A missing customer ID stays the empty string. The index is not checked.

## Part C: `analyze(cleaned) -> dict`

`cleaned` is a correct Part B table (the grader passes its own, so Part C is graded even if Part B has a bug). Return a dict with these keys:

| key | value |
|---|---|
| `"rows"` | rows in `cleaned` |
| `"imputed_rows"` | rows with `price_imputed == 1` |
| `"total_sales"` | sum of `line_amount` |
| `"top_country"` | country with the greatest total `line_amount` (ties: alphabetically first) |
| `"mean_invoice_total"` | mean over invoices of each invoice's total `line_amount` |
| `"outlier_invoices"` | number of invoice totals $> Q_3 + 1.5\,(Q_3 - Q_1)$, quartiles of the invoice totals by linear interpolation (the pandas default) |
| `"pct_repeat_customers"` | percent of identified customers (non-empty ID) with at least two distinct invoices |
| `"widest_price_product"` | product with the largest (max minus min) of its `unit_price` values (ties: alphabetically first) |

## Part D: `customer_summary(cleaned) -> DataFrame`

One row per non-empty `customer_id`, using only rows with a customer ID. Columns, in this order:

`customer_id` (string), `invoice_count` (distinct invoices), `product_count` (distinct products), `total_quantity` (sum of quantity), `total_spending` (sum of `line_amount`), `average_invoice_value` (`total_spending / invoice_count`), `favorite_category` (the category with the greatest total quantity for that customer; ties: alphabetically first), `spending_rank` (rank of `total_spending`, largest = 1, tied values share the smallest rank: 1, 1, 3). Sort by `customer_id` as strings. Dtypes: the counts, `total_quantity` and `spending_rank` are int64; `total_spending` and `average_invoice_value` are float64.

## Output: `save_outputs(cleaned, summary, cleaned_path, summary_path) -> None`

Write the two given tables to the two given paths as UTF-8 CSV with a header and no index column, decimals with `float_format="%.6f"`. Integers must be written as integers and an empty customer ID as an empty cell. (In the exam the paths are `problem3_cleaned_transactions.csv` and `problem3_customer_summary.csv`.)

Decimal answers in the dicts may be unrounded or rounded with `round(value, 4)`; the grader allows $10^{-6}$ (or the 4-decimal rounding). Counts and text must be exact. No function may modify its input DataFrames.

## Example

`transactions` (as CSV text):

```
record_id,invoice_id,product_code,description,quantity,unit_price,customer_id,country
1,536365,85123A,"MUG, RED",6,2.55,17850,United Kingdom
2,536365,71053,LAMP,6,3.39,17850, uk
3,536366,22633,"TIN ""STAR""",abc,1.85,13047,France
4,C536379,85123A,"MUG, RED",1,2.55,13047,France
5,536367,22633,"TIN ""STAR""",24,,12583,UK
6,536367,84879,CANDLE,3,0,12583,UK
7,536368,22633,"TIN ""STAR""",2.5,1.65,,
8,536368,71053,LAMP,4,,,
2,536365,71053,LAMP,6,3.39,17850, uk
9, c536369 ,85123A,"MUG, RED",2,2.55,  17850 ,United Kingdom
10,536370,99999,GIFT CARD,1,,9001,germany
11,536370,22633,"TIN ""STAR""",30,1.65,9001,germany
12,536371,84879,CANDLE,12,1.25,13047,  FRANCE
13,536371,85123A,"MUG, RED",3.0,2.95,13047,france
14,536371,23084,RABBIT LIGHT,2,4.15,13047,France
15,536372,71053,LAMP,1,3.39,13047 ,France
```

`products`: `85123A Kitchen`, `71053 Home`, `22633 Home`, `84879 Gifts`, `90210 Toys`.

`explore` gives `rows 16, columns 8, distinct_customers 4, duplicate_copies 1, nonnumeric_quantity 1, nonpositive_price 1, median_positive_price 2.55, pct_missing_customer 12.5, cancellation_rows 2`.

`clean` keeps 9 rows. Records 3 (quantity `abc`), 4 and 9 (cancellations), 6 (price 0), 7 (quantity 2.5) and the second copy of record 2 are dropped in steps 3 and 4. Record 10 is dropped in step 5: product 99999 has no price anywhere. Records 5 and 8 get imputed prices 1.65 and 3.39. In order, the rows are records `2, 1, 5, 8, 11, 14, 12, 13, 15`; the first two are

```
record_id invoice_id product_code description quantity unit_price customer_id country        category price_imputed line_amount quantity_band
2         536365     71053        LAMP        6        3.39       17850       United Kingdom Home     0             20.34       Medium
1         536365     85123A       MUG, RED    6        2.55       17850       United Kingdom Kitchen  0             15.30       Medium
```

Record 8 has an empty customer ID and country `Unknown`; record 14 has category `Unclassified`; record 13 has quantity 3 (`"3.0"` is a whole number).

`analyze` gives `rows 9, imputed_rows 2, total_sales 173.84, top_country "United Kingdom", mean_invoice_total 28.9733, outlier_invoices 0, pct_repeat_customers 25.0, widest_price_product "85123A"`.

`customer_summary` gives

```
customer_id invoice_count product_count total_quantity total_spending average_invoice_value favorite_category spending_rank
12583       1             1             24             39.60          39.60                 Home              2
13047       2             4             18             35.54          17.77                 Gifts             4
17850       1             2             12             35.64          35.64                 Home              3
9001        1             1             30             49.50          49.50                 Home              1
```

`"9001"` sorts last because the IDs are compared as strings. The first data line of the cleaned CSV is `2,536365,71053,LAMP,6,3.390000,17850,United Kingdom,Home,0,20.340000,Medium`.

**Restrictions.** Only `import numpy as np` and `import pandas as pd`; no standard-library imports.

**Constraints.** Up to about 20,000 transaction rows. 20 seconds of CPU time for all five functions together.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem3_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem3_submission.py
import numpy as np
import pandas as pd


def explore(transactions, products):
    # Your code
    return {}


def clean(transactions, products):
    # Your code
    return transactions


def analyze(cleaned):
    # Your code
    return {}


def customer_summary(cleaned):
    # Your code
    return cleaned


def save_outputs(cleaned, summary, cleaned_path, summary_path):
    # Your code
    pass


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem3_submission as submission
importlib.reload(submission)
# e.g. submission.clean(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNrFfQl34zaS8F/BOG+2yR6ZkeSzlTj7ud3ujifu49numWRlPQ4lUjLXFKkmKR/j9X//6gBAgKRkdyebze60KRAoFKoKdeHgw0YcbgzExrL7anPR3dnoiI15VAZQ9NB8U8ZlEmHZ8d0iyfI4nYkgDcUijxYB/cqhaZyIMCgD4Zx/2oKCqyjIiyBxsX0SFUWWAoB0mSTYU5wuy6iAgu0u/Azj6TSeLJPyHkugIErLHJ83JkkUpIRbFi6TyE+DOeGxyLNxEs23/GI5nsdFEWdUK0iS7DYK/Xi+yPIS4Q830uV8cY8vF4ByUGyM4HEcpGmEgxzKX36SZQusPw2SIsIRR0Xpl/E8ypaljy+2ENEymDFQhuXlURD6k+KmAu8VZW78KjMf+o/yeGIUhstFEk8CJEBVOMuz5WJ8b7bNg7SYZvncKANQs8j4DVWujZ9flkFaxklkY4AI4jiLbJlPiHxBGCzKKNzsd/t7m0UwXyTRJnCNm+VlH+qU+TJ6hN9TAFeQVFwFeQp89BbEmUv6v5+jZBHlhSivglJIpohZHoRYyFwQzr+meTYXsrkqffkv17tML9NDWV3EhZhkaQms/36ZxmXx/Y/456fvJVT4vYjDn77n2oCEJ05KEUbTGKAKZJeYLtNJCZJQCBSTkAr9l4xbGVxH8BRdppXECBYq8S8o+pcnjn89fP/p9PhcJHFR4pAiBovAClFcZbepgEZny1Q4N3ERA1buD+IcwZUiX0K30U2U31MjGNpbmBDLHFrmQVxEAn+K27i8EoEAeEUwi8RtHpcwYAFMFq+jdCCK4F7cIrq3QQFvs3TWESlCFSFIsbiKwxCqx+liWRZIPeZCKkk6yRb3+gfKLtJX/qRZIABqusDS78RRlgRjcfTpcyGCPBIFTpwcaZUiKi8K8b4vPuVZRxSZSLJJkCBEmOqFmC+BKOMIiQokynDqxvA6iZEOYxj/FbByGkzKLPewp77XReYGYrZE/jvccwGwQOrKK5xDolhEwDCoBbNgdpXcg7QkU9AxiMUky5HQeYREL0U8RQThJ01xOQ45U5Eo55+Oj9/4bw+PLj6eiQPsHQd8mU6SAHpHNjjHd5NogZLiDi5TAf8t4BXXAoECikehA7IYdsS8mKk60G+aIZHTUJbgfxVzHaxbAfHThXOn2oI2M9ooDmX55MooncL447Qog3QSOXcdfu9dRGmR5a7RnLqNymWeijsvBMU7uXJcb7JYwr/EZsflyhENU5xQd8d5nuUGFB6yAStdeEER5HlwD3hX4yiugkXkzLKyA0KZ4r8gnweXGyDZoLAvNxRmBdC6BMUWOThyqO961FTiQjSFOqoSwnI7Ynq58YAAH4HhBfclHgrQO9HdIpqAihIPRv1H6K7CbJJkxTrMOiIvs+SgF23udkQgH/cVvjCzbgFljWxQlPcw0GmSBYgYllOf9gtjMDMeHw7plh9XDIcHIKu7bWNTr2h4ugNgSFyAeotLaA14JInjCtAUKIXmu1v5zuweNWkQkyZMsU2cTjVwKcjI7ySRRARiSHLhP5Jc+I8peFGeA8WIEIjcPLjDP8G4cGZiU9y6rouwYaDxvyMRgR0VXZx8LVPFphMpOnETJKAfhJMEYOJAw0QosOIB/gy8rdmja/E+AkuXPEMq7UnFVRFpMI0o6R2XKApIBGWZy/eXG8SOCkiLuBgCUtX5XxcLUzQIf1/SAZBrTCaTqhoCcmVQxxiEAbGlwVcwQBk/wKu/5BZqWAuKLGbEhU9ThNiBT8VKflCHBkeqJg38sRRIIQG7nk/On+9b6MxA/SHnZK8vVKUXCqqNqV+kwQIsefnnquYkS6OvVMrQpZLKOxRJsu0bLrndOHttfJwwngAh0XFxTXzswVnYIUAnjKLFwQV4eoaASQQvgHx19JoAXMuEYJGHMOndnUV4YIsTdMT42+kePIvuXIXnBfX3rWQPkOwEp7BVgewo4E4KZ+y26RpoXumZ1uZaT4zlI/HWntk4gg5rO5Ds1OSVgkMQDOcl8+dLmBTg3TjTtCNegjqFmfHy5fUtPilM2HU8Agsgpin1PEUHFccAPg54gTOYXkEKLmM+gwAmBYlLxSIJJpGnvE6y+jifQC8Oq5kVuOTPBtgAexxxTfDO0HikTh0hfo1NYhBkGHDhYtOI4iYIkpx/xwuH21AfloCTPtHCJZXIPcQ5IIthhA5+PI3Rs0SXuRrLdw+x+JvoPf4gbrP8Gr36gIRXIPvAH9X6UhIZkK8ojE58nDpFhK4gYPVcKv9gkhi0q+lyt3jSFMuCPSzA94YRIIHI132hW4LzzjEUdvURQOQYgkGANC9wSOSHvw8mYgIdYBMYJHvdGK2QY4/OAUQXm+NlCJoUi8kHB7c8xgBquYDpJMrbeBIhQKBQDBY6m8oZDDgrihYkQlgyBX8JzTdCgBZLeOeJC3wR51AMg0leYBdoF7gzpHAUWmJVdtGbBAgeRJdTn8BEudI2T4hSWLY3Bh+llO6IHO+BkEwU3wszbKhE0kdRzHE2OH1T7oCJ0M2PBxJSTRWNIaa5roqCfkdc96E3Wz8Syp1aoTWSp0lBeCIh+kSGvlFMZICIzQnRxK4gh+lvmgOieVRmGckNBP9hOfD600fw8kHKoh9k1KWFmIQJph0BEssU4/kHSRrZjmJICOwUi1v+oy4hsoQweAqiNUMHloPMB8knAoSzY+0EzdMZzM4oVAyrAhzgZJjNgdrTAMTP1xUvKXu0LIIgznzKMFhZjndYAjMAMKuSBzEpjas4wTxYNsHMCATMMc4FrPn3848fANeUZk5RhoAhzdXFfXkFJZsnQvUjfpQpDj+M85/Ej1Uf8BJ+w6z8Hyorf2qE/DDZjPAfGHtXJvFYF/EfKPGWZZxUxZl+/O8iq5IEi6C8MpsX8SwNqmbFfVFLL6gfOdiFcTC51iUgxSkmCzjXUExiGGYaaXUHBddxuZlEUE30vF2vJ326YoB5ojnmCzukUiC6kMZHKb9FEGPK8TJVfXjTOAGJVj+dyw1AO8sjjABlpgU8Ue/l5/Q6xRxOkSWYUckoAwAdxgvkmvdSeoo/H58dw8yRtPA+wV/H9zEN5vuuB4oLm4NXtwhywOwyBap4WNkDaY3y0unCZAD3AcG4ALAt90W5IPEd2OovwUAcb3f7lAk7/efhb+f+68MPH47fAAYPlxu+zGT6Pg4GBqb4yT+ju2gin8DP56dJNl8Arpcbj5fp6cePn/wPH98cnwM4B2TFe5vBXMWHf4LgRvx4WNynE/3iFHzII4DBv84j48cbcDKrX++iFO1zlh/fLVwzxXLBuZinsix+AJHe3EEZW85B34DZivSMpVBRATKzEejZTPzJVTS5LpwinyCLy8C2t2c839EbBmOFdMyumTjzYgaE+QHUInqAnOOjaVOULzR3yBmSqUQ0VGUec1rRMlFshIajNm+2zCNw7IhKICUQXwOith96fg/x+R35oZiUi5r+4VCiPQBpoZS0xH6AarKg5jJEzlLWMw+Rh3/T7HGAz1Abgp9HiaDMjKNJAHp5oJhhntTS5Uij4UgiOg50OdnJUpkcSof6y4KAGeVjmA8+50plOYo4uIwwd+MQpgp6Yjk7CgwYIHB9ByMadLrjMS4LwHCuyXeLEg8FW3ZAKXqD5vOgnC8Ts2RK2RFQ1aibgfi3QXLtIDNqpttw07E6yzOHBvWoQruyWNEjdAdNAyZTKuAvOwFVQqd8jqkF/ulBb0BI9BydOXieoI4wmEPgc0KWGeEO2m2jwQsvCEPZh9usXPHGqOcViyRGdmOfw83eqKVhxTxvuQjR625pawaJyXoyvgWtVx8NMBR4RXSUiXcYfjWhWmhJLSQh4fl3ULFOQYBWo4JBOosCbRJQa7qKeN/UlMb5tUQ/VHOnPvaaQBAeONGeBfWDoZHXAYzD1aNqr9TWY2Ws6n3SxPeCxSJKQ0dlfLjR5QarPFbv/2nTq60XsoKv4/SjtGKHy9lhgTbI5dxKrYGXyXrvg/I9uIwNmWYd9JXIAWKGQBog0UmUoFaqf+UF8CxhkcdSmhBYz/vvDMNTUrCO0Y/rtmuXv6FFAcIoUA8vOuIFA1EzCifgC+gP4t7ZC0zFPrbnE9vwx5RFC/qFwr0C9p1eQRwHaYG0nVxJC0GuYGUgKP0cBRDE/kp/3R84Gk1ug/tik1d3IZIBvzDWgFAoJbQsTe47qk+IXCpT47AUcBJJulLodNyLJJpBaH6LbpOgmp7nuYYxA6dMhBkyhQFfxRRfMhccw2j+BxlHwwozutL4ghj+j3AMW/ofwvILDRGCHr5adACDQkj6IHyncFtkByC7v4fDaab60EuyFZcBc3vwPk8iP1uwP2ln27A6T7La1HvOgDGiBLn4f0LBxpDVmDfCYc9J9jDsjkzhbhn+15Cg6nXd4Em5NYZMpd82YHaTSNitgUIxTmMUatyckVIQq31HavXHjl+O7HFNpD7NYwCT3PvkyToAAgJdH6Mp26P/mEabhGcYFZM8pqhCLEBUS9qKwkkCqhCr5WsM2GC6RlivoMV6O8M0xrSKils9iJvhufTLsRN5vq9fQMgnnU0EDp7mlIz6FPsBEMCpKYafkfL5MPLTQ5Be6S2mTDBhhVqWcpNS6OBfdMcMr90qQlaQwGBtWk6r0JfExBUTWiGJjPUR9P4fH6jXR6yviD0HT9zRsRho2nqUi5FskM9uhtpFVENZWbM/MkJiSSl0wA+ErrKl3HWQeyjHhIOXZEFYOA4h8T2KDc4JSkVsuLypBjMZjlJ2ED4hRImMWUFRg7KJB6I5LzhgvJRbfS5pX09R/eTEDIusjKRRErHgQwYsMOEPK2gjRGdFKKonOnqwOAWLIWM1ImuBv2iWjEx3VoV0yp8tKO/Y6NbUEpS1cIicuDmkcLi6aeVZSlTKcwpIx5l3XmL65OSjIh4neTz+48hf5yfvDk/P3ndklC6rgjrzKbmrVoENXWZvmEJa9rsKFzs6pk0wVbIK2BnGeQSTj5nhAKZ1N0vhGJUxdpErNE8uTt4fn/lnx4enHY1dzc0pFhFKj50K87DUx9wMZ3YwO0lLJsaIjC1nLBpVYg7YZKiqWn/LcbM7CYs6xK4d/KfeEGYM1yuG+JoChhELfnNINIei3MMckM/NUPO430C4rp2bkGkXKyUhBVFOjhFrMx4jamDSaqRxyyy7pjUF8aD4MfC6mAFWqkv28jooIp0h4hxIlRJ7fXrc7fYqBL5qGKswlgtDhCauxOAOsUwySjpBa+yRntYN0G2EUnoF+wV5RomilXgHAvDtbrc7GH3zRKa9kXYeh4rM/R1cR60VqxVNGEuHX7lPM7eiVZhF7DTxZjuwLgji0Yzbv3IIs6+bk5cbvOkPZ6E2GXofoB633Ef49NybGZNvtmouMTRZS273k4ktFeVxFdQMauugcuKl2kNbg06DA4HKlBU7PqFuvwnyQvXhxWU0LxzaPZPaVoG0qlr3xwWUANBwpq6Z8AItgYuQ5JJgjzXnmcwxSn++BL3FQoHeCu0iSNXQagoX1XOcLiOTfxM2sKwPBwRkuDMYgf6mdWHAVmapBanHyw0JWlU2esO3mA3GsKctTJH6lhd9sKjrdR+/Zk2sufHhG6zO77U8cnGuqZW/SaVJJgwryuFMRce7uXejocGrxux78BwnZZ1k6Uyty9R1tnDilHeZkTffwQBiGuRCrQy6phKQndPiRi2n/TsHrHFG97plr8pX2ZI/ChdznwGtV4TPtCEVLC3gCC8HKQ6dVQu1HbHVZt6kOztSAVmuFdtXmyLdgH3VAwwdncp3XeeVklIZZ1niNBDjdwiqtCQXwZUWON1CBSgrTQpta54KFfCwdvN9jG9woWyglgcw3OGVXW0samu6OeEhDzrUDy544g0eZ4gLMeOlLsxn4GoeLhkXnHHSO9GCtLjF1Q0IryOuha9p9WPzEy35XqZ5NIWYLJ1EYhzB5Ono/d8xxGfIPjCsyT0uWfHe/Q7CSClih3A+ZBWGYCtHlNZDsa6kY7VG1VzqnYMI6h+ZsZQbzRdocFftFteLw9QP/P8iXL2s2aEtAB1zG1JH7pjp4MI8dqI32oNplCYuuqNDCMD+y/TiV//o46l8CwKd5aEfh3IBNL3J4kmkf4M7EC5BieMs5BIjQ8AFfCaivOdfeKbAB9mayOqTZVECy3INkaYdelOAydHp8eGH/zNk6jbDwI0qg0DOMvWLoGCmd1lGEh3MIfjBHNvYnftjYKUxwIvfPhEzHkz8KFOrWtagW+/MQeALigxbMDDegSk///zeIGyDC4q0kwr7irq6qMzKIPFtmnJZgdMJdwnUyXi5EUBwEsxgNLILUobcdhrcZDAloVuLtgqYj6dtiG6IvUG1GrY10lloW+/q+Fsva71W72py0RxyjQsrBmzz4/jXT6cfz479X45/U8Ke3RZqSiTLeSp/hHEBniEOSPJMlatDTT5vE+PSNEvl+SeTTU1viGousgLM8U1kTok5uGlB6re9WgASpAaBRgoZNTVAyYKvjPPO53EA0w4/HJ7+9l/tQ5Si7VclkqhBEumChW9NQFx40ESl6lwO1jaJUZb5VbFiwIg+uM5R0CDlbRyiTuRJJ+WHRvDm+Ojk/eGpGsHDN5KnMbRVI2lHEaTl4vjXCwOLBmnaR/BoKH//4lfiAREGWNHjhjtbu1u7O/y8v9Prbx3y8/vP7zri7PgN/9rlP31vR1bt7e3vdPnxM3qrofgFhh1mcyBbR/fRb/ax1+vubPHjKeBlgd/ytl41wIvltQVzy4Cp0OrvblHxi4uTD/D7/OLw7HLjBU3Fsdyv0/P2Fepb3e09fnybo9xa4Lf5zRHC33v1FF16DbqsBb5j4L73JO59iYuE3N/Zl2P//IsFdbcJdX97XyF/dPjhzemxJDD/6T4Fcc+AuP80np4avKe4rP41YO43YbbKwrYNoQHnf0OmZA0xIaivxFNs79fZLgRBF0/OiF63Qn9PPr/C//jx3cnbC3F0eGaLl6yGQRU9QfQ4D9J7G3CvCXgNz7a6dZY9Ad4g+15vrZCpqj2v35wUQrw9O/xwdGwD32oBvpL4W54anveq2cG0Oet6203w/a3uviw+O3z9+uRCnJ68+/nCYu+213vWrO4Z03qvv0Ykew2RRLiiCXhUae1PeKRyOLQJ8ktcTiBQwco0XavOfs7mkSrW7LeLDba9i6cUAVL5q25fSedFdo/FI478vhObv/M/PqYugzp0zWl5TAV5tCMYAhg0iAd7Xb0lEUP0ar8wJckCTu8B5kEeRqk1imqcFomM8cjd6hAiSCimYL1APkHdf5x8uDh8dyzniRDnxxfi41uxJWAOnZvSLn7+ePrm+Ez2cHwoLj7+8/i0I16ffqaZ8OLT4afjM6z+8+HJBwD1y8nFi6YveXT68egXBvL3w/fi74cS4uuTszfi7fExdSFRxziHUMdlSOBWtyv+Jrb2xEsRP1IqxEFlQdtwxF/FtlrDPHx9BEMcYnife/BvBDO8cLZdd8Qp0tjYc99XmU6gtZ/hCtbDZECEHyLMnZE6ODKxz4wQau6j2plYRLJlF7QA4mf33e+I7a7LvRMgai5bQyRLfoxM/tKr4dZgRyUqlimgEyTZbKl3RHKdncG+qoP+EBKKX7wadbiHYa87Gq3YDP8djGYG8VOEkX8Z82YyDP7ZtxKMU7HAxVCZMcmyhDqpjQMZMFHpXmytSLrk9LTco/KAyTUkzI2kxA3xoaJSrw8c7ojePtgH/OM+usDjoamrX+3v7WrxuMJUCJN92LRCppKhaaPUPOXT41wWny+CWE6bz0UAsOtyswtyYwwYxyQZdxPk4BvTEB+a/a9EanldOSE05z5fiyrShdrXXFtUWteeQmpU1IM5xKnxbBgec/DUxiKFNH8ShXfHZ+8PP/y2qmumGwE5Pjk7Nok5MqhJFQzKghjR80iTmalzLhX8kij/qI4aobZcxJNr565oOdh2V9RZlEQpVnXdkQnhtnCKjli0QHAI+JAWEljraccLJa6A/9VqVPVGfAg5l0c9HFf8KBaseQqz9y/lvdxCYPS/RB2vW1rLKEuA0/W6W+2HDu1drFX13fbqCnsdDpS3cpPgh+8lyXudnSoXagPdXw9UGs1Nafo3eyvg9Lrr4Wg/uls51P1VSPW21gPTXpKQbs6uUO4Zvhk1FgqF0kYModcRoKS3Ong7zE5H7HbEPqgg1EZ9VEW42QD+h7WgAl7Vsk3vu7ihzhJbikiJ8agba7uuJnLvuFL568dU45g9iqdFae/rROnVquqEQBtLes+SEwCtJKbJX93JX70+bl/6q8BD/2hEO2RVyaCNYD6229VN5JnBAcyuGNv1kSGYIgQ+37BF7IgvoBI6pMY7QqYUdJ78ALfkWPoC4KmVjyG+7AgNCgmh8+sF7eZhNSALO+x5sUn2MAN/xyIBg0SFRW/dkcRoxZk1C8+RHieao+vKiyFX0hK19EYuZnHciG4TUO+aVNc1IPBKaiyoAcaV/KrrrxKuuo7gHkHlKu4fyUyZ1KlENpjo9X3hK1QFg7NFtWWnoO5Vgu+a8CerRoFEldBpUiocmCYOKX3zxRa/oJGR/Te3HUjeGPAs04A4Gc2VyzBE34U2VyBAvNMpXjhuWzDtmtOldmK0NhtAO4Gz4jZW2sk1pO7Rg3Obr/1yDfo9iT9QmWdQjcy0gb4+ybTx6zQ0YocgkVwDbV0mdEUD7qw2PxlJvYvbcFxxhg7wBHFakBOLZyFlnrojAqES7psq4Y6VOlA/EzJrSM3CuvNrur0lRFM1t5c8YUScgobhZMRLhBx38VYK5Q9b3rv1RhmB0bBrOLTP7Iojv2/vijgGVhInP/nWOEqK5FXQrpJHvVc95X2jx+o22/efbN+33fEWGFsmDGnCtU0gS4AvRwrgluW8r4JH9FwJkN4+HyLBQPEY9kZWUozNm6Ed1pCqAtI1gfS+CshODROVZ+l/I5SuBeVpXGREDFPPjolxLtasJ8CWhojuQ9AzuODtOrQgIG/6o9nIdwHwgq7g1VBx8qYQt1cZmyvcDAiFeGFZPJ3igrhaC49zIVeCuELlEWgLjvlI/E96iPi4W8ld01xjecNc67Ryq3i2UE0p8BoqRhr0mzHpfxsm3wm9mCbvXEC/Bg8Xg+JIoDSnvV50p0IS/SCf6HYFvMMim9IBdMmeGDdt3ARJHAIQUKfcnipSaXXak0EcECGGO6Pai+E2b3qhmEX6b0s+j4kHbZ2q4mB7JBMD6E3CU+39zmBk7l/jdI83uUJho1iRBFVskn+Pd0kdoNMJfn317vvvRX8H6Ca3nh3QDjJTtjETQ93SYNAQx24tcKkZ0noAEw53ecC8qwd/GuCX1YG0cC3Yfh2s0ZIQDF1j2uJFJVTFaKXYkqKJjtXsvmuZ29j9HU3pA8C3aznyKNzygHrNLWH4MPtNAqMyAPqGejOONMaY/J10Kms3WpdrMg0eysGQppVOFqmkqcz3vuoqO1Tle40YBPHqaESqUw1l5tPR8cKxa9SOhOPmqk10YWC20B2D4uj8HwLdHtJvdAkhORjBvdzYE8x55ysegqOKHRHNF+W9wKXlgg+FUYFrHyy5I2cKXbrQww1EbxE7iZxcST+QW1xceTunQ1pESrGieCsYNboK1LBt30m1f2G0po/yjsGrW0wd83SAHAcKAZ4wOQDl3hHXUbTw1e0ZaVDDeA00OZznQ1O3Od0hR/+wxL+5H0vo7Vjy4qTl3Cnab6e60QceCnur/D9wK0X9NimJOkac1lhu5Fm2q+rqvhvpT3NdfVCHFvWN7Nqdkaa9UyikqOI4s2Z1c1cMU1KQI9pPDGanLyMn/QZ1K07IqiqoVNGvMFD3yEITiBbWooFng1SGD+G6EMh+kRvRcS82KBsaM9Avy50rt4FsPOIVAlA54ic9JEaZX/9NOFcAOkbQOAiqO4ISem3cDgEc9SO6odj2dhSpqMxOfg9zUOw6uEFlRsYVa470RsQJu/0oITlYxhX1Mqq0oJcLig24JYxuoc4dUhqCHCrxk+jaGu5BbUgZ4C5xY9vNAJNd7TtvBjSuh0ktA05q+NFt7H1qbtKBvoCQBCRH02GNrAVC64aegSiAND27MRlD9Br+IoNX2nlOJEQHQeZkWnuob2cx4T+Dsj8eiG4T7oqtMgM13eCFu3rfzACziSB/FSI2telqWJxFaaPjlt1IawjW09LoLcFNyOGvdSLgCI+RP9oyT7dkr7B/wG1cAFkMBAvJQq6VqXpysaSIMCV2jUpN3t7Rsa7V0Fiaio58yiqtgJ5vawYPnUor+6VayFQPlFQ5Fnxj/DJyD21ZHfmkaUbXUbVXfeC1nUFjJUflb/kNXRJ0ufHI6Z8qxUEPYE3xKnSzAxwvumfA0C+AMP5d1FzCmPw6JPGTxyywEl2akNcvTGCn6kbdAEt5I7rZ9Wa1oDzZHcD9otOjAO0LTR5++gtr7y/PgrJqHj7ZFkVOh13tEsVYNMVKX1VzT3vMkMWPNYlF6HaOHxW4iWv96jaGhUcDpGcC+qo/omM9epP9UC/+gu6g9cVrrUZu2Dpcd3j9VMGTR3we2y4PWoktb0qEmt3WIbSgT2/B8ksPHNFrOdTS5ILe4TAc7JCrji2HCGqEP8FImifWbOSMsxVfCMS2URW3GlMYdT4HP5pTmCRkOzqd+x4It5wbr/pdtVXgFO8Ctq6/NQ6f54gnqDeaqMwkzM/SJVR06yMdQ5J4gl4C7Y2U6xBOo+ouQQ9dGuc6uj9Igvk4DEQ+EE5OyRuGiobRXXffHKjgIA2S+39HDqlifftweoPbKTsoBVqTsO1iUcaFAZKfx47xv5pUEEh79QChDhHFEWtP/C3JgGh3va5LXOt1TVZoJKDpHjWtimTrvZWtGe3nzgsti7ujugZQw7ZB7QIoMjsua7/qDDoMrVDRvBoq32qsz4d/gdj1yxZU0n4rNurQqhSIQKN0TyXIx7hLFc+1YYbBIEYNPqeb9USHujfo6+LlirXpLqlkz3bqpZC9SFj1Lpo+IPpkLE3NDcrsP+TDV6OamLiN/b2qaq/brFv3VKx9vAMa33VdmykSGWcWb+gKWiJlm9vV3FvMOCE30GPCgdJzo21zJ7XpOJGGI+kAHErwp0EEIDDwcCHSgedNkAu3CbR9U3PTv7tR/h1Lq+IX3c4REeMhWumrAeiKzQ7bN0O3kVfJRitti7rXB6jOg/zeVjnBZNJmBxs6hP2JtvnZsA14awOAbc5XPvlATGGHkVMRVsEXOr1J5xnkOU6Zn6AXD4/mXSPBUAIcVTrgByplsKq4r4q/UH6RTQ6XFLrEUl74ivscge7bJ91nlEnlt4/Kj1Xf9miVoSbnWQbBQBXTOZKEGk6MnmFKyYlv9leb/AQ7uJH3tdoyYTarywUAN3N8QXqN5hi3yGgpHhMM4J4pv2NFqZ800dpNLOb6ULorzrhVgWQKlChedDS46knODwsCjLVD6I7W3g1j5c0KdaO7JHg4XZ9iwyZGZhUGwvoCYShCmgljXrvjv+qbD3TiZnebXZOSV/boDI7yTygphBXscYTTPyxjxYf/4gKvUOIbMPHMIwQZY/D4je8fhNEkBg+rPX810/krvEnfymA5+p73jpHNatmURck56wMSsstm1Zn6nIAy4LgtET/Dgeo52nwFahY/GmF8PUHdZa4+JYFv0BPEpniHniznI7I84G2XahA8Atvr23D1Z0LwghYfL8g36QXuHghLEoyjZO23CbCd/DQBVX4cVBev4zvNepkyQG1xXW0AwW54U4f0yAHoyPzGB5TKlnYnChwAcMBKPsjfj9Yqn+7C1uxUrI/o1C/wa45yeD2iO2v5rC4XKBbSOxOxh2v9aQh4Z38dIpCrfnhjrzxltv4qKf4qjC3R1KPumsZinntyn0Tn9iqeXGEhfQHD/mQMyUJJlygYwmCoFykTEDeE4OokHVL1Psaha8XEVESrxEVXsD/zQo4tAPGk5nKR9IyRCUa+pYu9HhptrI9iYONKUFAksWoegVH2eb0yzDPruw+zGIO0IS4E37FDS+tLsyFTQS3F3LZXI2bZNUEKpY0E0C6mEuRPAOEOVn4TRg2Wv11CSTFKSeG1WeIquMGhgzGZuY9yPagaNJbzR4Jcm740tgPEvcYXhI7UlGup3CcvXjsPmu0GOGOpy7qNju6bJ7tCKSO+Vyq3ZL9mP1qnAd4ogsaHVkhcXXuXNiU0RQH4zB9KCflLKVVbUxgkALelQ+6Nv1CHn/LQU94qtT4utAKD1k/N6D1kcuDyrOmzh65t67cPvm6etT0E9+MW3S2bArw0pb4A1SCF+bplENbnmm6eIpv+QJNS9C3Usz73oz/VRLhX37Z6VX3bavf5XGq/hh8a1z+/VPvO0w1a5xvXNT7JtEa9z274YAs7UI79zRomJJkcuSiW4qKXq5Z+TDU0GY1ayIIUtmGSEava3awmCH3E7ipLQrlrpUn5cUDKLjY/TTJu+TQJMYRlhfeeB6juxm0Io7UHsGu5RIoMKoGKMxjm0KdNaKU+KGlrzwPUwQUbvktWfm7L/tpRvkz9SVBEfPGTvTyg7Rx+77E4wEukeOFMLiBjvCiPj3NmS12BRjEnrkEoT5NWaFEDrlqIl+sQCNGXO4BXLlpIOwPk5PfouJJzXzXviOpaBvVMZ/Gr66+MwVBSJrBv2TU8QqCMp1YMeSS0q6S2jNgR5vF4k31Cd2T2Lqm3um/2QLBzpoLuuhr5ymFa3Rt8Mi+naN+uDOzDazrG96K6uULT3ueMu4aCF33UpwYNTsvDM0irsqLVsORnoySVraypL2ltntO3Bqt7NtHREtmGDvULNbQYNfImqlN1HwU/mfLUyjV1X4WC1TZA1XkrcJuLNXBrONi4KUOx1HjRxjrphVZ41p1i3LGBlxQpn7gZGmXyOxfRHX4flSrX9GsBHpoPQTx+nhRCAb59bpJHuJFNbVfTGOH1qrUdI9z/UztE8A47EFYY3sHlxrKcbu63ONR0q6vpUVMpDk2XmjQ2R0EfhShuxFWkvkv70ASJu9wePc+zXNBGF6qWMHff/GfNQWWFP7lmNJUb22nHqbIQ0OCRT7KudoWbrquJnjlRhPb9LKcVPbPbdresZuuhezb2f/V2ebfjTZVGvR2t9RNkW+zsxm1vRR8pJJlhn4xo7ZrrTM+01hitQX8rTDVb0/qiGvUcDrs1m74ieEE+GUadAlE+hxLIQDXGQFXeC/wAyKgCvLlMcXHNd5LAS1O79xCketZJoIL35xKnfPyANH6K8VJy5hJlQnkJf0CCiq/t4q14xk1OdOudOjpt+SHVBSDV86cz/pyNvATK8GEkTDbGBkx9HZnT68lTYFv1o0Edc99idbzbTJmu9ZCAqfKLyzTNHgrak/pIu6+bXpNbx5pNeyvOdFINcO7/KTj3azgrp6GBsTLHrThvAc5b/T8D360avpUH2qRx3RC3or4NqG//Kahv11CvDHoDddNYWrNlJWo7vT/bl/4j/CfUy+qKOW9+HeKz0twLYMwCv4ynfAv6/EAoV5LwUxBbjHsU4p3saRHILzGBjqXV/XXtaqIh2+iLzT2LAQ0HrvLfdAFjq3hQeU0922+XEyyqviJp1O2bfqEZT9Wkg7/x7i9T+S1OU0JaQq6XWkh2e2pVyfwSqBHndGR7U5Yi/jKPXV8Ox6pdT37K1o0EqKQBh9lrMqBIC0u4WJLt4TwlpXXEpb5gSq+oU5cOXdniAiVDQC2GPn7W8nmzdK+nLg/BcyPu10XJ9G3rmyi/p0+tWCtU8gp8O2JsHptuC2gndAepjsYmfHl5M4qZWB/YlN857dOabYWUJdMcEkF92ndSZ2NtiKv5uDrYbbvlgLjCSndF/Pv7Qt2Nx8f/D6p7Ufo=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
